# 10 — Decision Tree Export for Deployment

**Goals**
- Export the final decision tree structure (`export_text` + `plot_tree`)
- Translate the tree into C if/else pseudocode
- Produce `decision_tree_c_reference.txt` for firmware


In [ ]:
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import export_text, plot_tree

MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'temperature_C', 'humidity_pct', 'soil_moisture_pct',
    'moisture_trend', 'light_lux', 'hour_of_day'
]

dt = joblib.load(MODELS_DIR / 'decision_tree_full.joblib')
le = joblib.load(MODELS_DIR / 'label_encoder.joblib')
print('Classes:', list(le.classes_))
print('\nFeature importances:')
for f, imp in zip(FEATURE_COLS, dt.feature_importances_):
    print(f'  {f:20s}: {imp:.3f}')


In [ ]:
# Full textual export
tree_txt = export_text(dt, feature_names=FEATURE_COLS, show_weights=True)
print(tree_txt)

with open(RESULTS_DIR / 'decision_tree_export_text.txt', 'w') as f:
    f.write(tree_txt)
print('\nSaved → results/decision_tree_export_text.txt')


In [ ]:
# Visual plot
plt.figure(figsize=(16, 10))
plot_tree(dt,
          feature_names=FEATURE_COLS,
          class_names=list(le.classes_),
          filled=True, rounded=True, fontsize=8)
plt.title('Decision Tree (full data)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / '10_decision_tree_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/10_decision_tree_plot.png')


## Manual translation to C-style if/else

The cell below walks the trained sklearn tree and emits ready-to-copy C code.


In [ ]:
def tree_to_c(tree, feature_names, class_names, node=0, indent=1):
    t = tree.tree_
    ind = '    ' * indent
    if t.feature[node] == -2:                       # leaf
        class_id = t.value[node].argmax()
        return f'{ind}return "{class_names[class_id]}";\n'
    feat = feature_names[t.feature[node]]
    thr  = t.threshold[node]
    left  = tree_to_c(tree, feature_names, class_names, t.children_left[node],  indent+1)
    right = tree_to_c(tree, feature_names, class_names, t.children_right[node], indent+1)
    code  = f'{ind}if ({feat} <= {thr:.4f}) {{\n'
    code += left
    code += f'{ind}}} else {{\n'
    code += right
    code += f'{ind}}}\n'
    return code

c_body = tree_to_c(dt, FEATURE_COLS, list(le.classes_))
c_code = '''/* Auto-generated decision-tree reference for firmware
 * Source: sklearn DecisionTreeClassifier trained on full labeled_dataset
 * Features must be supplied in the same units as training data.
 */
const char* predict_urgency(float temperature_C, float humidity_pct,
                            float soil_moisture_pct, float moisture_trend,
                            float light_lux, float hour_of_day) {
''' + c_body + '}\n'

print(c_code)
with open(RESULTS_DIR / 'decision_tree_c_reference.txt', 'w') as f:
    f.write(c_code)
print('Saved → results/decision_tree_c_reference.txt')
